# Telco Customer Churn — STEP 03. 특성 공학 비교 실험

**목표**: 결측치 처리 3종, 인코딩 2종, 스케일링 3종을 비교하고 파생 변수 3개를 만든다. 간단한 Logistic Regression 성능 비교는 **임시 검증 지표**이며, STEP 04/05에서 변수 선택과 모델별 상세 평가를 진행한다.

- 같은 train/test 분할을 유지해 전략을 비교한다.
- 결측치 대체, 인코딩, 스케일링은 **훈련 데이터에만 fit**하고 테스트에는 transform만 한다(데이터 누수 방지).
- `customerID`는 식별자이므로 제외한다.
- 결측치 처리 방식 `drop`은 관측치 수가 달라지므로 다른 실험과 직접 비교할 때 주의한다. 여기서는 평균·중앙값·최빈값을 사용한다.

## 1. 준비 및 데이터 로드

In [ ]:
# 패키지 설치: %pip install pandas numpy matplotlib seaborn scikit-learn
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

local_path = Path('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
if local_path.exists():
    df = pd.read_csv(local_path)
    print('로컬 CSV:', local_path)
else:
    df = pd.read_csv(url)
    print('IBM 공개 CSV를 사용했습니다.')

# 문자열 공백을 NaN으로 변환하되 원본 df는 별도로 유지
working = df.copy()
working['TotalCharges'] = pd.to_numeric(working['TotalCharges'].astype(str).str.strip(), errors='coerce')
X = working.drop(columns=['customerID', 'Churn'])
y = working['Churn'].map({'No': 0, 'Yes': 1}).astype(int)
print('X:', X.shape, 'y:', y.shape)
print('TotalCharges 결측치:', X['TotalCharges'].isna().sum())


## 2. 파생 변수 3종 생성

- `TenureGroup`: 가입 기간을 네 단계로 구분한다.
- `ServiceCount`: 가입한 서비스 개수를 센다.
- `AvgChargeRatio`: 누적 요금을 예상 누적 요금(`MonthlyCharges × tenure`)으로 나눈 값이다. 가입 0개월에는 결측치로 유지한 후 학습 데이터의 통계량으로 대체한다. 이 변수는 유용성을 검증해야 하는 실험용 파생 변수다.

파생 변수는 입력 행에만 의존하므로 테스트 타겟 정보는 이용하지 않는다.

In [ ]:
SERVICE_COLUMNS = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                   'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

def add_features(frame):
    frame = frame.copy()
    frame['TenureGroup'] = pd.cut(frame['tenure'], bins=[-1, 12, 24, 48, np.inf],
                                  labels=['0-12m', '13-24m', '25-48m', '49m+']).astype('object')
    frame['ServiceCount'] = sum(frame[c].eq('Yes').astype(int) for c in SERVICE_COLUMNS)
    denominator = frame['MonthlyCharges'] * frame['tenure']
    frame['AvgChargeRatio'] = (frame['TotalCharges'] / denominator.replace(0, np.nan))
    frame['AvgChargeRatio'] = frame['AvgChargeRatio'].replace([np.inf, -np.inf], np.nan)
    # 가입 기간 0개월에는 계산 불가능하므로 NaN으로 남겨 뒤의 결측치 처리기에 맡긴다.
    return frame

example = add_features(X)
display(example[['tenure', 'MonthlyCharges', 'TotalCharges',
                 'TenureGroup', 'ServiceCount', 'AvgChargeRatio']].head(8))
print('생성된 파생 변수:', sorted(set(example.columns) - set(X.columns)))


## 3. 공통 훈련/테스트 분할

모든 전략에서 동일한 난수 시드와 stratified 분할을 사용한다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('훈련:', X_train.shape, '테스트:', X_test.shape)
print('훈련 이탈 비율:', round(y_train.mean()*100, 2), '%')
print('테스트 이탈 비율:', round(y_test.mean()*100, 2), '%')


## 4. 조립식 전처리 파이프라인

`OrdinalEncoder`는 과제의 Label Encoding 비교에 대응하는 **입력 X용 인코더**다. 순서 없는 범주에 인위적 순위를 부여할 수 있으므로 결과 해석에 주의한다. 범주 결측값은 최빈값으로 대체하며, `TotalCharges` 등 수치 특성에는 지정한 결측치 전략을 적용한다.

In [ ]:
NUMERIC_ORIGINAL = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

def make_preprocessor(X_sample, missing='mean', encoding='onehot', scaling='standard'):
    num_cols = X_sample.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_sample.select_dtypes(exclude=np.number).columns.tolist()

    num_steps = [('imputer', SimpleImputer(strategy=missing))]
    if scaling == 'standard':
        num_steps.append(('scaler', StandardScaler()))
    elif scaling == 'minmax':
        num_steps.append(('scaler', MinMaxScaler()))
    elif scaling == 'robust':
        num_steps.append(('scaler', RobustScaler()))
    elif scaling != 'none':
        raise ValueError(f'알 수 없는 scaling: {scaling}')

    if encoding == 'onehot':
        encoder = OneHotEncoder(handle_unknown='ignore')
    elif encoding == 'ordinal':
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    else:
        raise ValueError(f'알 수 없는 encoding: {encoding}')
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                         ('encoder', encoder)])
    return ColumnTransformer([
        ('num', Pipeline(num_steps), num_cols),
        ('cat', cat_pipe, cat_cols)
    ], remainder='drop')

def evaluate(label, missing='mean', encoding='onehot', scaling='standard', derived=True):
    a = add_features(X_train) if derived else X_train.copy()
    b = add_features(X_test) if derived else X_test.copy()
    pre = make_preprocessor(a, missing, encoding, scaling)
    pipe = Pipeline([('preprocess', pre),
                     ('model', LogisticRegression(max_iter=3000, random_state=42))])
    pipe.fit(a, y_train)
    pred = pipe.predict(b)
    prob = pipe.predict_proba(b)[:, 1]
    result = {
        '실험':label, '결측치':missing, '인코딩':encoding, '스케일링':scaling,
        '파생 변수':derived, 'Accuracy':accuracy_score(y_test, pred),
        'Precision':precision_score(y_test, pred, zero_division=0),
        'Recall':recall_score(y_test, pred, zero_division=0),
        'F1':f1_score(y_test, pred, zero_division=0),
        'ROC-AUC':roc_auc_score(y_test, prob)
    }
    return result, pipe


## 5. 결측치 처리 3종 비교

인코딩, 스케일링, 파생 변수 설정은 고정하고 **수치형 결측치 처리 방법만** 변경한다.

In [ ]:
missing_results = []
for method in ['mean', 'median', 'most_frequent']:
    result, _ = evaluate('Missing-' + method, missing=method)
    missing_results.append(result)
display(pd.DataFrame(missing_results).round(4))


## 6. 인코딩 2종 비교

One-Hot Encoding과 Ordinal Encoding을 비교한다. 다른 조건은 동일하게 유지한다.

In [ ]:
encoding_results = []
for method in ['onehot', 'ordinal']:
    result, _ = evaluate('Encoding-' + method, encoding=method)
    encoding_results.append(result)
display(pd.DataFrame(encoding_results).round(4))


## 7. 스케일링 3종 비교

StandardScaler, MinMaxScaler, RobustScaler를 비교한다.

In [ ]:
scaling_results = []
for method in ['standard', 'minmax', 'robust']:
    result, _ = evaluate('Scaling-' + method, scaling=method)
    scaling_results.append(result)
display(pd.DataFrame(scaling_results).round(4))


## 8. 파생 변수 전/후 비교

같은 설정에서 새 변수 추가 여부만 변경한다.

In [ ]:
feature_results = []
for derived in [False, True]:
    result, _ = evaluate('Derived-' + str(derived), derived=derived)
    feature_results.append(result)
display(pd.DataFrame(feature_results).round(4))


## 9. 탐색적 전처리 조합 비교

본 노트북의 조합 비교는 특성 선택 전의 탐색 실험이다. Logistic Regression의 Base에는 평균 대체와 One-Hot을 적용하므로 무전처리 기준선과 구분한다. RF 중요도 기반 변수 선택은 STEP04에서 수행한다. 선택을 포함한 조합 비교는 `complete_assignment.py`, 동일 모델 기준선 비교와 중첩 CV는 `robust_evaluation.py`에서 수행한다.

In [ ]:
configs = [
    ('Base', 'mean', 'onehot', 'none', False),
    ('Exp-1', 'mean', 'onehot', 'standard', True),
    ('Exp-2', 'median', 'ordinal', 'minmax', True),
    ('Exp-3', 'most_frequent', 'onehot', 'robust', True),
]
comparison = []
for name, missing, enc, scaling, derived in configs:
    result, _ = evaluate(name, missing, enc, scaling, derived)
    comparison.append(result)
comparison_df = pd.DataFrame(comparison)
display(comparison_df.round(4))


## 10. 결과 시각화 및 저장

성능 차이를 그래프와 CSV로 저장해 이후 보고서에 활용한다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ['F1', 'ROC-AUC']):
    ax.bar(comparison_df['실험'], comparison_df[metric])
    ax.set_title(metric + ' by preprocessing experiment')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    for i, v in enumerate(comparison_df[metric]):
        ax.text(i, v + 0.015, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
output = Path('results')
output.mkdir(exist_ok=True)
fig.savefig(output / 'step03_performance.png', dpi=160, bbox_inches='tight')
plt.show()
comparison_df.to_csv(output / 'step03_comparison.csv', index=False, encoding='utf-8-sig')
print('저장 완료:', output / 'step03_comparison.csv', output / 'step03_performance.png')


## 11. 비교 결과의 해석 범위

- 개별 전처리 비교는 나머지 조건을 고정한다.
- 파생 변수 비교는 3개를 동시에 추가한 결과이며 변수별 개별 효과와 구분한다.
- 여러 설정이 함께 달라지는 조합 차이를 단일 처리의 효과로 해석하지 않는다.
- 테스트 분할은 STEP05에서도 사용되므로 점수는 탐색적 결과로 해석한다.
- 동일 모델 비교와 중첩 CV는 `robust_evaluation.py`에서 기존 테스트셋을 제외하여 수행한다.